<a href="https://colab.research.google.com/github/pjastr-uwm/fakultet_io_2026/blob/main/lecture6/creating_corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Tworzenie korpusów językowych dla języka polskiego

Ten notatnik pokazuje, jak za pomocą **NLTK** i kilku bibliotek pomocniczych stworzyć proste korpusy językowe w pięciu popularnych formatach:

| Format | Opis | Zastosowanie |
|--------|------|--------------|
| **Plain text** | Surowy tekst, jeden dokument = jeden plik | Proste korpusy |
| **JSONL** | Jedna linia = jeden obiekt JSON | Metadane + tekst |
| **CoNLL** | Kolumny tab-separated (token, POS, NER…) | Anotacja sekwencyjna |
| **Parquet** | Format kolumnowy Apache Arrow | Duże zbiory, HF Datasets |
| **TEI/XML** | Standard humanistyki cyfrowej | Korpusy historyczne |



## 0. Instalacja i importy

In [1]:
# Instalacja bibliotek
!pip install -q nltk pyarrow pandas lxml

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

import json
import os
import pandas as pd
from datetime import datetime
from lxml import etree

print("✅ Wszystkie biblioteki gotowe!")

✅ Wszystkie biblioteki gotowe!


## 1. Dane źródłowe — przykładowe teksty po polsku

Używamy kilku krótkich tekstów w języku polskim. Każdy tekst ma przypisane **metadane** (autor, źródło, gatunek), które wykorzystamy w formatach JSONL i TEI/XML.

In [2]:
# Źródłowe dokumenty korpusu
dokumenty = [
    {
        "id": "dok_001",
        "tytul": "Jesień w Tatrach",
        "autor": "Anna Kowalska",
        "gatunek": "opis",
        "data": "2024-10-15",
        "tekst": (
            "Jesień w Tatrach to pora niezwykłych kolorów. Złote modrzewie kontrastują "
            "z ciemną zielenią świerków, a szczyty gór pokrywają pierwsze opady śniegu. "
            "Szlaki turystyczne są mniej zatłoczone, a powietrze kryształowo czyste. "
            "Wędrówka Doliną Kościeliską w październiku to prawdziwa uczta dla zmysłów."
        )
    },
    {
        "id": "dok_002",
        "tytul": "Przepis na bigos",
        "autor": "Jan Nowaćki",
        "gatunek": "przepis",
        "data": "2024-11-02",
        "tekst": (
            "Bigos to tradycyjna polska potrawa, której przygotowanie wymaga cierpliwości. "
            "Główne składniki to kiszona kapusta, wędzonki i suszone grzyby. "
            "Potrawę gotuje się na małym ogniu przez kilka godzin, a najlepiej smakuje "
            "odgrzewana następnego dnia. Dawniej bigos podawano na szlacheckich ucztach."
        )
    },
    {
        "id": "dok_003",
        "tytul": "Algorytmy sortowania",
        "autor": "Maria Wiśniewska",
        "gatunek": "naukowy",
        "data": "2024-09-20",
        "tekst": (
            "Algorytmy sortowania stanowią podstawę informatyki. Najprostszym przykładem "
            "jest sortowanie bąbelkowe, które porównuje sąsiednie elementy i zamienia je "
            "miejscami. Bardziej wydajne metody, takie jak quicksort czy mergesort, "
            "osiągają złożoność O(n log n) w przeciętnym przypadku."
        )
    },
    {
        "id": "dok_004",
        "tytul": "Poranek nad Wisłą",
        "autor": "Piotr Zieliński",
        "gatunek": "proza",
        "data": "2024-08-05",
        "tekst": (
            "Mgła unosiła się nad Wisłą jak delikatna zasłona. Pierwsze promienie słońca "
            "przebiły się przez chmury i oświetliły nadbrzeżne wierzby. Rybak stojący "
            "na drewnianym pomoście zarzucił wędkę i czekał cierpliwie. Cisza poranka "
            "przerywana była jedynie śpiewem ptaków i szumem wody."
        )
    },
    {
        "id": "dok_005",
        "tytul": "Historia Gdańska",
        "autor": "Ewa Lewandowska",
        "gatunek": "historia",
        "data": "2024-07-12",
        "tekst": (
            "Gdańsk należy do najstarszych miast nad Bałtykiem. Jego początki sięgają "
            "X wieku, kiedy powstał tu ważny gród obronny. W późnym średniowieczu miasto "
            "rozkwitło dzięki handlowi morskiemu i członkostwu w Hanzie. Dziś Gdańsk "
            "jest ważnym ośrodkiem kultury i turystyki."
        )
    }
]

print(f"📄 Załadowano {len(dokumenty)} dokumentów:")
for d in dokumenty:
    print(f"   • [{d['gatunek']}] {d['tytul']} — {d['autor']}")

📄 Załadowano 5 dokumentów:
   • [opis] Jesień w Tatrach — Anna Kowalska
   • [przepis] Przepis na bigos — Jan Nowaćki
   • [naukowy] Algorytmy sortowania — Maria Wiśniewska
   • [proza] Poranek nad Wisłą — Piotr Zieliński
   • [historia] Historia Gdańska — Ewa Lewandowska


## 2. Tokenizacja i tagowanie z NLTK

Używamy NLTK do:
- **tokenizacji** na zdania i słowa (`sent_tokenize`, `word_tokenize`),
- **tagowania części mowy** (`pos_tag`) — uwaga: tagger NLTK jest trenowany na języku angielskim, więc tagi będą przybliżone. Dla pełnej precyzji w polskim warto użyć np. `spaCy` z modelem `pl_core_news_sm`.

Tworzymy funkcję pomocniczą, która przetworzy każdy dokument.

In [3]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag


def analizuj_dokument(tekst):
    """
    Tokenizuje tekst na zdania i słowa, a następnie przypisuje
    tagi POS za pomocą NLTK.

    Zwraca listę zdań, gdzie każde zdanie to lista krotek (token, tag_POS).
    """
    zdania = sent_tokenize(tekst, language='polish')
    wynik = []
    for zdanie in zdania:
        tokeny = word_tokenize(zdanie, language='polish')
        tagi = pos_tag(tokeny)  # tagger angielski — przybliżone wyniki
        wynik.append(tagi)
    return wynik


# Przetwórz wszystkie dokumenty
przetworzone = {}
for dok in dokumenty:
    przetworzone[dok['id']] = analizuj_dokument(dok['tekst'])

# Podgląd wyniku dla pierwszego dokumentu
print(f"\n🔍 Podgląd analizy dokumentu «dok_001»:\n")
for i, zdanie in enumerate(przetworzone['dok_001']):
    print(f"  Zdanie {i+1}:")
    for token, tag in zdanie:
        print(f"    {token:20s} → {tag}")
    print()


🔍 Podgląd analizy dokumentu «dok_001»:

  Zdanie 1:
    Jesień               → NNP
    w                    → VBZ
    Tatrach              → NNP
    to                   → TO
    pora                 → VB
    niezwykłych          → JJ
    kolorów              → NN
    .                    → .

  Zdanie 2:
    Złote                → NNP
    modrzewie            → NN
    kontrastują          → NNP
    z                    → NNP
    ciemną               → NN
    zielenią             → NN
    świerków             → NNP
    ,                    → ,
    a                    → DT
    szczyty              → NN
    gór                  → NN
    pokrywają            → NN
    pierwsze             → NN
    opady                → JJ
    śniegu               → NN
    .                    → .

  Zdanie 3:
    Szlaki               → NNP
    turystyczne          → NN
    są                   → NN
    mniej                → NN
    zatłoczone           → NN
    ,                    → ,
    a            

---

## 3. Katalog wyjściowy

In [4]:
KATALOG = "korpus_polski"
os.makedirs(KATALOG, exist_ok=True)

for podkatalog in ['plain_text', 'jsonl', 'conll', 'parquet', 'tei_xml']:
    os.makedirs(os.path.join(KATALOG, podkatalog), exist_ok=True)

print(f"📁 Struktura katalogów:")
for root, dirs, files in os.walk(KATALOG):
    level = root.replace(KATALOG, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")

📁 Struktura katalogów:
📂 korpus_polski/
  📂 conll/
  📂 parquet/
  📂 tei_xml/
  📂 plain_text/
  📂 jsonl/


---

## 4. Format 1: Plain Text

Najprostrzy format — każdy dokument to osobny plik `.txt`.

**Zalety:** łatwość odczytu, kompatybilność ze wszystkimi narzędziami.  
**Wady:** brak metadanych, brak struktury anotacyjnej.

In [5]:
katalog_txt = os.path.join(KATALOG, 'plain_text')

for dok in dokumenty:
    sciezka = os.path.join(katalog_txt, f"{dok['id']}.txt")
    with open(sciezka, 'w', encoding='utf-8') as f:
        f.write(dok['tekst'])

print("✅ Zapisano pliki Plain Text:\n")
for plik in sorted(os.listdir(katalog_txt)):
    sciezka = os.path.join(katalog_txt, plik)
    rozmiar = os.path.getsize(sciezka)
    print(f"   {plik:20s}  ({rozmiar} bajtów)")

# Podgląd zawartości
print("\n🔍 Podgląd dok_001.txt:")
print("-" * 60)
with open(os.path.join(katalog_txt, 'dok_001.txt'), 'r', encoding='utf-8') as f:
    print(f.read())

✅ Zapisano pliki Plain Text:

   dok_001.txt           (318 bajtów)
   dok_002.txt           (301 bajtów)
   dok_003.txt           (291 bajtów)
   dok_004.txt           (297 bajtów)
   dok_005.txt           (282 bajtów)

🔍 Podgląd dok_001.txt:
------------------------------------------------------------
Jesień w Tatrach to pora niezwykłych kolorów. Złote modrzewie kontrastują z ciemną zielenią świerków, a szczyty gór pokrywają pierwsze opady śniegu. Szlaki turystyczne są mniej zatłoczone, a powietrze kryształowo czyste. Wędrówka Doliną Kościeliską w październiku to prawdziwa uczta dla zmysłów.


### Odczyt korpusu Plain Text za pomocą NLTK

NLTK udostępnia klasę `PlaintextCorpusReader`, która pozwala wygodnie wczytać taki korpus.

In [6]:
from nltk.corpus.reader import PlaintextCorpusReader

# Wczytanie korpusu
korpus_txt = PlaintextCorpusReader(katalog_txt, r'.*\.txt', encoding='utf-8')

print("Pliki w korpusie:", korpus_txt.fileids())
print(f"\nŁączna liczba słów: {len(korpus_txt.words())}")
print(f"Łączna liczba zdań: {len(korpus_txt.sents())}")

# Częstotliwość słów
fd = nltk.FreqDist(w.lower() for w in korpus_txt.words() if w.isalpha())
print("\n📊 10 najczęstszych słów:")
for slowo, ile in fd.most_common(10):
    print(f"   {slowo:15s} → {ile}")

Pliki w korpusie: ['dok_001.txt', 'dok_002.txt', 'dok_003.txt', 'dok_004.txt', 'dok_005.txt']

Łączna liczba słów: 221
Łączna liczba zdań: 19

📊 10 najczęstszych słów:
   i               → 7
   w               → 5
   to              → 4
   a               → 3
   się             → 3
   na              → 3
   pierwsze        → 2
   bigos           → 2
   przez           → 2
   jest            → 2


---

## 5. Format 2: JSONL (JSON Lines)

Każda linia pliku to niezależny obiekt JSON. Format popularny w uczeniu maszynowym — łatwy do strumieniowego przetwarzania i wzbogacania o metadane.

**Zalety:** metadane obok tekstu, łatwy streaming, kompatybilność z HuggingFace.  
**Wady:** brak natywnej kompresji, redundancja kluczy.

In [7]:
sciezka_jsonl = os.path.join(KATALOG, 'jsonl', 'korpus.jsonl')

with open(sciezka_jsonl, 'w', encoding='utf-8') as f:
    for dok in dokumenty:
        # Dodajemy statystyki z NLTK
        zdania = przetworzone[dok['id']]
        tokeny_all = [token for zd in zdania for token, _ in zd]

        rekord = {
            "id": dok['id'],
            "metadane": {
                "tytul": dok['tytul'],
                "autor": dok['autor'],
                "gatunek": dok['gatunek'],
                "data": dok['data']
            },
            "tekst": dok['tekst'],
            "statystyki": {
                "liczba_zdan": len(zdania),
                "liczba_tokenow": len(tokeny_all),
                "unikalne_tokeny": len(set(tokeny_all))
            },
            "tokeny": tokeny_all
        }
        f.write(json.dumps(rekord, ensure_ascii=False) + '\n')

print("✅ Zapisano plik JSONL\n")

# Podgląd
print("🔍 Podgląd pierwszego rekordu (sformatowany):")
print("-" * 60)
with open(sciezka_jsonl, 'r', encoding='utf-8') as f:
    pierwszy = json.loads(f.readline())
    # Wyświetl bez listy tokenów (dla czytelności)
    pokaz = {k: v for k, v in pierwszy.items() if k != 'tokeny'}
    pokaz['tokeny'] = f"[{len(pierwszy['tokeny'])} tokenów...]"
    print(json.dumps(pokaz, ensure_ascii=False, indent=2))

✅ Zapisano plik JSONL

🔍 Podgląd pierwszego rekordu (sformatowany):
------------------------------------------------------------
{
  "id": "dok_001",
  "metadane": {
    "tytul": "Jesień w Tatrach",
    "autor": "Anna Kowalska",
    "gatunek": "opis",
    "data": "2024-10-15"
  },
  "tekst": "Jesień w Tatrach to pora niezwykłych kolorów. Złote modrzewie kontrastują z ciemną zielenią świerków, a szczyty gór pokrywają pierwsze opady śniegu. Szlaki turystyczne są mniej zatłoczone, a powietrze kryształowo czyste. Wędrówka Doliną Kościeliską w październiku to prawdziwa uczta dla zmysłów.",
  "statystyki": {
    "liczba_zdan": 4,
    "liczba_tokenow": 46,
    "unikalne_tokeny": 39
  },
  "tokeny": "[46 tokenów...]"
}


---

## 6. Format 3: CoNLL

Format kolumnowy rozdzielany tabulatorem. Każdy wiersz to jeden token z anotacją; puste linie oddzielają zdania, a linie zaczynające się od `# doc_id =` oddzielają dokumenty.

Kolumny w naszym korpusie:
1. `TOKEN` — forma słownikowa
2. `POS` — część mowy (z NLTK)
3. `NER` — etykieta encji nazwanej (uproszczona, `O` = brak)

**Zalety:** standard w NLP, obsługiwany przez wiele narzędzi.  
**Wady:** brak metadanych, ograniczony do sekwencji tokenów.

In [8]:
def proste_ner(token, pos_tag_val):
    """
    Prosta heurystyka NER dla polskiego tekstu.
    Prawdziwy NER wymagałby dedykowanego modelu (np. spaCy, Stanza).
    """
    # Znane nazwy własne z naszego korpusu
    lokalizacje = {
        'Tatrach', 'Tatry', 'Doliną', 'Kościeliską',
        'Wisłą', 'Gdańsk', 'Bałtykiem', 'Hanzie'
    }
    osoby = {'Anna', 'Kowalska', 'Jan', 'Nowaćki', 'Maria', 'Wiśniewska',
             'Piotr', 'Zieliński', 'Ewa', 'Lewandowska'}

    if token in lokalizacje:
        return 'B-LOC'
    elif token in osoby:
        return 'B-PER'
    elif token[0].isupper() and pos_tag_val in ('NNP', 'NN', 'JJ'):
        return 'B-MISC'
    return 'O'


sciezka_conll = os.path.join(KATALOG, 'conll', 'korpus.conll')

with open(sciezka_conll, 'w', encoding='utf-8') as f:
    for dok in dokumenty:
        for zdanie in przetworzone[dok['id']]:
            for token, tag in zdanie:
                ner = proste_ner(token, tag)
                f.write(f"{token}\t{tag}\t{ner}\n")
            f.write('\n')  # pusta linia między zdaniami

print("✅ Zapisano plik CoNLL\n")

# Podgląd
print("🔍 Podgląd (pierwsze 30 linii):")
print("-" * 60)
with open(sciezka_conll, 'r', encoding='utf-8') as f:
    for i, linia in enumerate(f):
        if i >= 30:
            print("  ...")
            break
        print(linia, end='')

✅ Zapisano plik CoNLL

🔍 Podgląd (pierwsze 30 linii):
------------------------------------------------------------
Jesień	NNP	B-MISC
w	VBZ	O
Tatrach	NNP	B-LOC
to	TO	O
pora	VB	O
niezwykłych	JJ	O
kolorów	NN	O
.	.	O

Złote	NNP	B-MISC
modrzewie	NN	O
kontrastują	NNP	O
z	NNP	O
ciemną	NN	O
zielenią	NN	O
świerków	NNP	O
,	,	O
a	DT	O
szczyty	NN	O
gór	NN	O
pokrywają	NN	O
pierwsze	NN	O
opady	JJ	O
śniegu	NN	O
.	.	O

Szlaki	NNP	B-MISC
turystyczne	NN	O
są	NN	O
mniej	NN	O
  ...


### Odczyt CoNLL z NLTK

NLTK ma klasę `ConllCorpusReader`, która pozwala wczytać takie pliki i operować na nich jak na natywnym korpusie.

In [9]:
from nltk.corpus.reader import ConllCorpusReader

# Definiujemy kolumny: słowo, POS, NER (chunk)
kolumny = ['words', 'pos', 'chunk']
korpus_conll = ConllCorpusReader(
    os.path.join(KATALOG, 'conll'),
    ['korpus.conll'],
    kolumny,
    separator='\t',
    encoding='utf-8'
)

print(f"Liczba zdań w korpusie CoNLL: {len(korpus_conll.sents())}")
print(f"Przykładowe zdanie (tagged):")
for token, tag in korpus_conll.tagged_sents()[0]:
    print(f"   {token:20s} {tag}")

Liczba zdań w korpusie CoNLL: 19
Przykładowe zdanie (tagged):
   Jesień               NNP
   w                    VBZ
   Tatrach              NNP
   to                   TO
   pora                 VB
   niezwykłych          JJ
   kolorów              NN
   .                    .


---

## 7. Format 4: Parquet (Apache Arrow)

Kolumnowy format binarny, wydajny dla dużych zbiorów danych. Używany m.in. przez **HuggingFace Datasets**.

**Zalety:** kompresja, szybki odczyt kolumnowy, obsługa typów danych.  
**Wady:** binarny (nie da się otworzyć w edytorze tekstu).

In [10]:
# Budujemy DataFrame z danymi korpusu
rekordy = []
for dok in dokumenty:
    zdania = przetworzone[dok['id']]
    tokeny_all = [token for zd in zdania for token, _ in zd]
    tagi_all = [tag for zd in zdania for _, tag in zd]

    rekordy.append({
        'id': dok['id'],
        'tytul': dok['tytul'],
        'autor': dok['autor'],
        'gatunek': dok['gatunek'],
        'data': dok['data'],
        'tekst': dok['tekst'],
        'liczba_zdan': len(zdania),
        'liczba_tokenow': len(tokeny_all),
        'tokeny': tokeny_all,
        'tagi_pos': tagi_all
    })

df = pd.DataFrame(rekordy)

sciezka_parquet = os.path.join(KATALOG, 'parquet', 'korpus.parquet')
df.to_parquet(sciezka_parquet, engine='pyarrow', index=False)

print("✅ Zapisano plik Parquet\n")

# Odczyt i podgląd
df_odczyt = pd.read_parquet(sciezka_parquet)
print("🔍 Schemat danych:")
print(df_odczyt.dtypes)
print(f"\nRozmiar pliku: {os.path.getsize(sciezka_parquet):,} bajtów")
print(f"\n📊 Podsumowanie:")
print(df_odczyt[['id', 'tytul', 'gatunek', 'liczba_zdan', 'liczba_tokenow']].to_string(index=False))

✅ Zapisano plik Parquet

🔍 Schemat danych:
id                object
tytul             object
autor             object
gatunek           object
data              object
tekst             object
liczba_zdan        int64
liczba_tokenow     int64
tokeny            object
tagi_pos          object
dtype: object

Rozmiar pliku: 10,868 bajtów

📊 Podsumowanie:
     id                tytul  gatunek  liczba_zdan  liczba_tokenow
dok_001     Jesień w Tatrach     opis            4              46
dok_002     Przepis na bigos  przepis            4              46
dok_003 Algorytmy sortowania  naukowy            3              43
dok_004    Poranek nad Wisłą    proza            4              43
dok_005     Historia Gdańska historia            4              43


---

## 8. Format 5: TEI/XML

**Text Encoding Initiative** — standard XML stosowany w humanistyce cyfrowej i korpusach historycznych. Zawiera bogaty nagłówek (`teiHeader`) z metadanymi oraz ustrukturyzowany tekst (`body`).

**Zalety:** bogata struktura metadanych, standard międzynarodowy, walidacja XML.  
**Wady:** złożoność, duży narzut składniowy.

In [11]:
TEI_NS = 'http://www.tei-c.org/ns/1.0'
XML_NS = 'http://www.w3.org/XML/1998/namespace'
NSMAP = {None: TEI_NS}


def tei_el(tag, **kwargs):
    """Tworzy element w przestrzeni nazw TEI."""
    return etree.SubElement(kwargs.pop('parent'), f'{{{TEI_NS}}}{tag}', **kwargs)


def buduj_tei_korpus(dokumenty, przetworzone):
    """
    Buduje pełny dokument TEI/XML z nagłówkiem i tekstem.
    """
    # Korzeń
    tei = etree.Element(f'{{{TEI_NS}}}TEI', nsmap=NSMAP)
    tei.set(f'{{{XML_NS}}}lang', 'pl')

    # === teiHeader ===
    header = etree.SubElement(tei, f'{{{TEI_NS}}}teiHeader')

    # fileDesc
    file_desc = tei_el('fileDesc', parent=header)
    title_stmt = tei_el('titleStmt', parent=file_desc)
    title = tei_el('title', parent=title_stmt)
    title.text = 'Przykładowy korpus języka polskiego'
    resp_stmt = tei_el('respStmt', parent=title_stmt)
    resp = tei_el('resp', parent=resp_stmt)
    resp.text = 'Przygotowanie korpusu'
    name = tei_el('name', parent=resp_stmt)
    name.text = 'Generator NLTK'

    publication = tei_el('publicationStmt', parent=file_desc)
    pub_p = tei_el('p', parent=publication)
    pub_p.text = 'Korpus przygotowany w celach edukacyjnych.'

    source = tei_el('sourceDesc', parent=file_desc)
    source_p = tei_el('p', parent=source)
    source_p.text = f'{len(dokumenty)} dokumentów w języku polskim.'

    # encodingDesc
    encoding = tei_el('encodingDesc', parent=header)
    tag_decl = tei_el('tagsDecl', parent=encoding)
    tag_usage = tei_el('tagUsage', parent=tag_decl, gi='w')
    total_tokens = sum(
        len([t for zd in przetworzone[d['id']] for t, _ in zd])
        for d in dokumenty
    )
    tag_usage.set('occurs', str(total_tokens))

    # === text / body ===
    text_el = etree.SubElement(tei, f'{{{TEI_NS}}}text')
    body = etree.SubElement(text_el, f'{{{TEI_NS}}}body')

    for dok in dokumenty:
        # Każdy dokument to <div>
        div = tei_el('div', parent=body)
        div.set('type', dok['gatunek'])
        div.set(f'{{{XML_NS}}}id', dok['id'])

        # Nagłówek dokumentu
        head = tei_el('head', parent=div)
        head.text = dok['tytul']

        # Informacja o autorze
        bibl = tei_el('bibl', parent=div)
        author = tei_el('author', parent=bibl)
        author.text = dok['autor']
        date = tei_el('date', parent=bibl, when=dok['data'])
        date.text = dok['data']

        # Zdania z tokenami
        for zdanie in przetworzone[dok['id']]:
            s = tei_el('s', parent=div)
            for token, tag in zdanie:
                w = tei_el('w', parent=s, pos=tag)
                w.text = token
                w.tail = ' '  # spacja między tokenami

    return tei


# Generuj i zapisz
tei_root = buduj_tei_korpus(dokumenty, przetworzone)
drzewo = etree.ElementTree(tei_root)

sciezka_tei = os.path.join(KATALOG, 'tei_xml', 'korpus_tei.xml')
drzewo.write(
    sciezka_tei,
    pretty_print=True,
    xml_declaration=True,
    encoding='UTF-8'
)

print("✅ Zapisano plik TEI/XML\n")

# Podgląd
print("🔍 Podgląd (pierwsze 50 linii):")
print("-" * 60)
with open(sciezka_tei, 'r', encoding='utf-8') as f:
    for i, linia in enumerate(f):
        if i >= 50:
            print("  ...")
            break
        print(linia, end='')

✅ Zapisano plik TEI/XML

🔍 Podgląd (pierwsze 50 linii):
------------------------------------------------------------
<?xml version='1.0' encoding='UTF-8'?>
<TEI xmlns="http://www.tei-c.org/ns/1.0" xml:lang="pl">
  <teiHeader>
    <fileDesc>
      <titleStmt>
        <title>Przykładowy korpus języka polskiego</title>
        <respStmt>
          <resp>Przygotowanie korpusu</resp>
          <name>Generator NLTK</name>
        </respStmt>
      </titleStmt>
      <publicationStmt>
        <p>Korpus przygotowany w celach edukacyjnych.</p>
      </publicationStmt>
      <sourceDesc>
        <p>5 dokumentów w języku polskim.</p>
      </sourceDesc>
    </fileDesc>
    <encodingDesc>
      <tagsDecl>
        <tagUsage gi="w" occurs="221"/>
      </tagsDecl>
    </encodingDesc>
  </teiHeader>
  <text>
    <body>
      <div type="opis" xml:id="dok_001">
        <head>Jesień w Tatrach</head>
        <bibl>
          <author>Anna Kowalska</author>
          <date when="2024-10-15">2024-10-15</dat